# 01 - Information Extraction - OpenAI

In this notebook, we explore ways that OpenAI LLMs can be used for extracting information relevant to infectious disease modeling, such as categorical keywords (e.g. diseases, treatments, populations, etc.), from publication titles/abstracts. This information will be used later for publication search, clustering, etc.

## Notebook Setup

Install the OpenAI Python SDK and `python-dotenv` for loading the API key from a local `.env` file.

In [ ]:
%pip install --upgrade --quiet openai python-dotenv

Load the OpenAI API key (and any other secrets) from `.env` into the environment so the SDK picks it up automatically.

In [ ]:
import dotenv
from genscai import paths

dotenv.load_dotenv(paths.root / ".env")

Load the publications from the database, skipping any publications without abstracts.

In [ ]:
import json
from genscai import paths

with open(paths.data / "training_modeling_papers.json", "r") as f:
    papers = json.load(f)

len(papers)

Define a prompt template that instructs the model to extract domain-specific keyword categories (disease, treatment, intervention, vector) from a paper abstract and return them as JSON.

In [ ]:
KEYWORD_PROMPT_TEMPLATE = """
Your goal is to identify important keywords in scientific paper abstracts.
For the abstract below, identify all diseases, treatments, interventions, and vectors mentioned.
List the keywords identified in a JSON array, with each item in the array including keyword_type and value.
The only valid keyword types are disease, treatment, intervention, and vector.
Only return the JSON array.

abstract:
{abstract}
"""

Run the prompt against the first paper using `gpt-4o`. The output is free-form text matching the JSON shape we asked for, but with no schema enforcement.

In [ ]:
from openai import OpenAI

client = OpenAI()

article = papers[0]

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": KEYWORD_PROMPT_TEMPLATE.format(abstract=article["abstract"]),
        },
    ],
)

print(completion.choices[0].message.content)

Using Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs

Re-run the same extraction using OpenAI's [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) API. By passing a Pydantic model as `response_format`, the SDK constrains the model to produce parseable, schema-valid JSON and returns a typed object.

In [ ]:
from pydantic import BaseModel


class Keyword(BaseModel):
    type: str
    value: str


class KeywordResults(BaseModel):
    keywords: list[Keyword]


completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        # {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": KEYWORD_PROMPT_TEMPLATE.format(abstract=article["abstract"]),
        }
    ],
    response_format=KeywordResults,
)

print(completion.choices[0].message.parsed)

Define a second prompt template for binary classification: does the abstract describe an infectious-disease modeling technique, and if so, which one?

In [ ]:
MODEL_CLASSIFICATION_PROMPT_TEMPLATE = """
Given the following scientific publication abstract,
identify if the publication references an infectious disease modeling technique.
Only return YES or NO.
If YES, also return the name of the technique or techniques used.

abstract:
{abstract}
"""

Apply the classification prompt to a small batch of papers and print each abstract alongside the model's verdict.

In [ ]:
for paper in papers[5:10]:
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": MODEL_CLASSIFICATION_PROMPT_TEMPLATE.format(abstract=paper["abstract"]),
            }
        ],
    )

    print(paper["abstract"])
    print(completion.choices[0].message.content)